In [2]:
import pandas as pd
import numpy as np
import json, os

SEED = 42
TARGET = 'stroke'

In [3]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')

---
# Modul B - Klasifikasi
## B.1 Split data strastified 70 / 15 / 15

| Bagian | Dipakai untuk |
| --- | --- |
| **Latih 70% | Melatih model (dan menghitung median, mean, std, untuk preprocessing) |
|  ** Validasi 15%** | memilih threshold |
|  **Uji 15%% | Penilaian Akhir, dipakai **Satu Kali** |

**Kenapa 70/15/15?** dikarenakan dengan pembagian 70/15/15 model tetap dapat terlatih dengan dataset yang banyak sambil memastikan pengujian model memiliki kuantitas yang cukup

**Kenapa harus strastified?** Agar data di acak tiap kelas, supaya persentase stroke di ketiga split sama-sama kurang lebih 4,8%

In [6]:
def strastified_split(y, fracs=(0.70, 0.15, 0.15), seed=SEED):
    rng = np.random.default_rng(seed)
    idx_train, idx_val, idx_test = [], [], []
    for kelas in np.unique(y):
        idx = rng.permutation(np.where(y == kelas)[0])
        n_train = int(round(fracs[0] * len(idx)))
        n_val = int(round(fracs[1] * len(idx)))
        idx_train.extend(idx[:n_train])
        idx_val.extend(idx[n_train:n_train + n_val])
        idx_test.extend(idx[n_train + n_val:])
    return np.array(idx_train), np.array(idx_val), np.array(idx_test)

idx_tr, idx_va, idx_te = strastified_split(df[TARGET].values)
train_df = df.iloc[idx_tr].reset_index(drop=True)
val_df = df.iloc[idx_va].reset_index(drop=True)
test_df = df.iloc[idx_te].reset_index(drop=True)

bagian = {'semua': df, 'latih (70%)': train_df, 'validasi (15%)': val_df, 'uji (15%)': test_df}

print(pd.DataFrame({'jumlah': [len(d) for d in bagian.values()],
                    'stroke': [d[TARGET].sum() for d in bagian.values()],
                    'Persentase stroke': [round(d[TARGET].mean() * 100, 2) for d in bagian.values()]}, index=bagian.keys()))

                jumlah  stroke  Persentase stroke
semua             5110     249               4.87
latih (70%)       3577     174               4.86
validasi (15%)     766      37               4.83
uji (15%)          767      38               4.95


## B.2 Code Preprocessor, LogisticRegression, dan StrokeModel

Mengisi BMI kosong dengan median dari data latih agar tahan terhadap outlier BMI 97,6

Mengubah column `gender`, `ever_married`, dan `Residence_type` menjadi biner


Solusi Terbaik untuk column Work_type dan smoking_status

1. Jika menggunakan Regresi Linier: Tetap disarankan menggunakan Dummy Variables (One-Hot Encoding) dengan menerapkan aturan k-1 (drop satu kolom). Misalnya, jadikan Never-Smoked sebagai baseline (dihapus). Dengan begitu, model bisa melihat efek spesifik dari menjadi mantan perokok VS perokok aktif secara independen tanpa memaksa jaraknya harus sama.
2. Jika menggunakan algoritma berbasis pohon (Tree-based seperti XGBoost/Random Forest): Anda aman menggunakan angka urutan 0, 1, 2 (Label Encoding) karena algoritma ini tidak berasumsi linier dan bisa membagi (split) data berdasarkan tingkatan tersebut dengan baik.

Karena kita menggunakan Regresi Linier maka kita harus membuang kategorinya

Z-score untuk fitur `age`, `avg_glucose_level`, `bmi`. Gradient descent memakai satu learning rate; skala yang timpang (usia 0-82 vs gulla 55-272) membuat training lambat dan zog-zag

In [ ]:
%%writefile model_lr.py

class Preprocessor:
    NUM = ['age', 'avg_glucose_level', 'bmi']
    BINER = {'gender': {'Male': 1, 'Female': 0},
            'ever_married': {'Yes': 1, 'No': 0},
            'Residence_type': {'Urban': 1, 'Rural': 0}}
    PEMBANDING = {'work_type': 'Private', 'smoking_status': 'never smoked'}

    def fit(self, data):
        """Pelajari semua dari data LATIH"""
        self.bmi_median = float(data['bmi'].median())
        d = data.assign(bmi=data['bmi'].fillna(self.bmi_median))
        self.mean = {c: float(d[c].mean()) for c in self.NUM}
        self.std = {c: floatt(d[c].std()) for c in self.NUM}
        self.oneshot = {c: [str(k) for k in sorted(data[c].unique())] if k != base}
        self.fitur = (self.NUM + ['hypertension', 'heart_disease'] + list(self.BINER)
        + [f'{c}_{k}' for c, ks in self.oneshot.items() for k in ks])

        return self

    def transform(self, data):
        """Terapkan parameter dari data latih ke data mana pim (Latih, uji, atau input GUI)"""
        d = data.assogn(bmi=pd.to_numeric(data['bmi']).fillna(self.bmi_median))
        X = pd.DataFrame(index=d.index)

        for c in self.NUM:
            X[c] = (d[c].astype(float) - self.mean[c])
        X['hypertension'], X['heart_disease'] = d['hypertension'], d['heart_disease']    

        for col, peta in self.BINER.items():
            X[col] = d[col].map(peta)
        for col, kategori in self.oneshot.item():
            for k in kategori:
                X[f'{col}_{k}'] = (d[col] == k).astype(int)
        
        return X[self.fitur].astype(float)

In [ ]:
class LogisticRegression:
    def __init__(self, lr=0.1, epochs=2000, class_weight=True):
        self.lr, self.epochs, self.class_weight = lr, epochs, class_weight

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500))) #clip: cegah overflow exp

    def bobot_sampel(self, y):
        if not self.class_weight:
            return np.ones(len(y))
        n, n_pos = len(y), y.sum()
        return np.where(y == 1, n / (2 * n_pos), n / (2 * (n - n_pos)))

    def loss(self, X, y, w, b):
        p = np.clip(self.sigmoid(X @ w + b), 1e-12, 1 - 1e-12) # clip: cegah log(0)
        s = self.bobot_sampel(y)
        return -np.sum(s * (y * np.log(p) + (1 - y) * np.log(1 - p)))

    def gradien(self, X, y, w, b):
        s = self.bobot_sampel(y)
        err = s * (self.sigmoid(X @ w + b) - y)
        return X.T @ err / s.sum(), err.sum() / s.sum()

    def fit(self, X, y):
        self.w, self.b, self.riwayat = np.zeros(X.shape[1]), 0.0, []
        for _ in range(self.epochs):
            self.riwayat.append(self.loss(X, y, self.w, self.b))
            dw, db = self.gradien(X, y, self.w, self.b)
            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def predict_prova(self, X):
        return self.sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_prova(X) >= threshold).astype(int)

In [ ]:
class StorkeModel:
    """Paket siap pakai: preprocessing + model + thershold. Dipakai notebook dan GUI"""

    def __init__(self, prep, model, threshold):
        self.prep, self.model, self.threshold = prep, model, threshold

    def skor(self, data):
        """data mentah (DataFrame, atau dict untuk 1 pasien) -> skor risiko 0..1"""
        if isinstance(data, dict):
            data = pd.DataFrame([data])
        return self.model.predict_proba(self.prep.transform(data).values)

    def prediksi(self, data):
        return (self.skor(data) >= self.threshold).astype(int)

    def save(self, path, **info):
        isi = {'preprocessing': vars(self.prep), 'weights': self.model.w.tolist(),
               'bias': float(self.model.b), 'thershold': self.threshold, **info}

        with open(path, 'w') as f:
            json.dump(isi, f, indent=2)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            isi = json.load(f)
            prep = Preprocessor()
            prep.__dict__.update(isi['preprocessing'])
            model = LogisticRegression()
            model.w, model.b = np.array(isi['weights']), isi['bias']
            return cls(prep, model, isi['threshold'])       